In [ ]:
import json
import random
import numpy as np

import nltk
nltk.download('punkt')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from keras.optimizers import SGD
data_file = open('BooksDatasetClean_yourgpt.json').read()
intents = json.loads(data_file)
words=[]
classes = []
documents = []
ignore_words = ['?', '!']
for intent in intents['intents']:
    for pattern in intent['patterns']:

        # take each word and tokenize it
        w = nltk.word_tokenize(pattern)
        words.extend(w)

        # adding documents
        documents.append((w, intent['tag']))

        # adding classes to our class list
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))

classes = sorted(list(set(classes)))

print (len(documents), "documents")
print (len(words), "unique lemmatized words")
print (len(classes), "classes", classes)
training = []
output_empty = [0] * len(classes)
for doc in documents:

    # initializing bag of words
    bag = []

    # list of tokenized words for the pattern
    pattern_words = doc[0]

    # lemmatize each word - create base word, in attempt to represent related words
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]

    # create our bag of words array with 1, if word match found in current pattern
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)

    # output is a '0' for each tag and '1' for current tag (for each pattern)
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1

    training.append([bag, output_row])

# shuffle our features and turn into np.array
random.shuffle(training)
training = np.array(training)

train_x = list(training[:, 0])
train_y = list(training[:, 1])

# Create model - 3 layers. First layer 128 neurons, second layer 64 neurons and 3rd output layer contains number of neurons
# equal to number of intents to predict output intent with softmax
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))

# Compile model. Stochastic gradient descent with Nesterov accelerated gradient gives good results for this model
sgd = SGD(lr=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

#fitting and saving the model
hist = model.fit(np.array(train_x), np.array(train_y), epochs=500, batch_size=5, verbose=1)
model.save('chatbot_model.h5', hist)

print("model created")

def clean_up_sentence(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]

    return sentence_words
def bow(sentence, words, show_details=True):
    # tokenize the pattern
    sentence_words = clean_up_sentence(sentence)

    # bag of words - matrix of N words, vocabulary matrix
    bag = [0] * len(words)
    for s in sentence_words:
        for i, w in enumerate(words):
            if w == s:

                # assign 1 if current word is in the vocabulary position
                bag[i] = 1
                if show_details:
                    print ("found in bag: %s" % w)

    return(np.array(bag))
def predict_class(sentence, model):
    # filter out predictions below a threshold
    p = bow(sentence, words,show_details=False)
    res = model.predict(np.array([p]))[0]
    ERROR_THRESHOLD = 0.25
    results = [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD]

    # sort by strength of probability
    results.sort(key=lambda x: x[1], reverse=True)
    return_list = []
    for r in results:
        return_list.append({"intent": classes[r[0]], "probability": str(r[1])})

    return return_list

def getResponse(ints, intents_json):
    tag = ints[0]['intent']
    list_of_intents = intents_json['intents']
    for i in list_of_intents:
        if(i['tag']== tag):
            result = random.choice(i['responses'])
            break

    return result
def chatbot_response(msg):
    ints = predict_class(msg, model)
    res = getResponse(ints, intents)
    return res

chatbot_response('Recommend a book in History')


In [ ]:
import pandas as pd
df=pd.read_json("BooksDatasetClean_yourgpt.json")
print(df.columns)

In [19]:
import pandas as pd
import re
from transformers import BertTokenizer

# Load dataset
data = pd.read_json("BooksDatasetClean_yourgpt.json")

# Preprocess text
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.lower().strip()  # Convert to lowercase
    return text

# Apply preprocessing to relevant columns
data_cleaned = data.copy()
data_cleaned['Title'] = data_cleaned['Title'].apply(preprocess_text)
data_cleaned['Authors'] = data_cleaned['Authors'].apply(preprocess_text)
data_cleaned['Description'] = data_cleaned['Description'].apply(preprocess_text)

# Combine columns for training input and output
data_cleaned['input_text'] = "Describe the book titled '" + data_cleaned['Title'] + \
                             "' by " + data_cleaned['Authors'] + "."
data_cleaned['output_text'] = data_cleaned['Description']


In [20]:
from transformers import BertForQuestionAnswering, BertTokenizer
import torch

# Load BERT model and tokenizer
model_name = "bert-base-uncased"
model = BertForQuestionAnswering.from_pretrained(model_name)
tokenizer = BertTokenizer.from_pretrained(model_name)

# Tokenize data for input and output (context and question)
contexts = list(data_cleaned['Description'])
questions = ["What is this book about?"] * len(contexts)

# Encode the inputs and outputs
inputs = tokenizer(questions, contexts, max_length=512, truncation=True, padding=True, return_tensors="pt")
labels = inputs['input_ids']  # Placeholder for labels in a QA task

# Fine-tuning for Question Answering
from torch.utils.data import Dataset, DataLoader

class QA_Dataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.inputs.input_ids[idx]),
            "attention_mask": torch.tensor(self.inputs.attention_mask[idx]),
            "labels": torch.tensor(self.labels[idx]),
        }

dataset = QA_Dataset(inputs, labels)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Fine-Tuning the Model
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(3):  # Adjust the number of epochs as needed
    for batch in dataloader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        print(f"Epoch: {epoch}, Loss: {loss.item()}")


ImportError: 
BertForQuestionAnswering requires the PyTorch library but it was not found in your environment.
However, we were able to find a TensorFlow installation. TensorFlow classes begin
with "TF", but are otherwise identically named to our PyTorch classes. This
means that the TF equivalent of the class you tried to import would be "TFBertForQuestionAnswering".
If you want to use TensorFlow, please use TF classes instead!

If you really do want to use PyTorch please go to
https://pytorch.org/get-started/locally/ and follow the instructions that
match your environment.


In [ ]:
from transformers import ReformerForConditionalGeneration, ReformerTokenizer
import torch

# Load Reformer model and tokenizer
model_name = "google/reformer-crime-and-punishment"
model = ReformerForConditionalGeneration.from_pretrained(model_name)
tokenizer = ReformerTokenizer.from_pretrained(model_name)

# Tokenize data for input and output (book titles as inputs and descriptions as outputs)
inputs = tokenizer(list(data_cleaned['input_text']), max_length=512, truncation=True, padding=True, return_tensors="pt")
outputs = tokenizer(list(data_cleaned['output_text']), max_length=512, truncation=True, padding=True, return_tensors="pt")

# Create Dataset and DataLoader
class BookDataset(Dataset):
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def __len__(self):
        return len(self.inputs.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.inputs.input_ids[idx]),
            "attention_mask": torch.tensor(self.inputs.attention_mask[idx]),
            "labels": torch.tensor(self.outputs.input_ids[idx]),
        }

dataset = BookDataset(inputs, outputs)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Fine-Tuning the Model
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(3):  # Adjust the number of epochs as needed
    for batch in dataloader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        print(f"Epoch: {epoch}, Loss: {loss.item()}")
